# Supply Chain Resilience Analysis

This notebook demonstrates resilience queries for the Supply Chain Knowledge Graph:
1. Single Points of Failure Analysis
2. What-If Scenario Analysis
3. Risk Assessment
4. Supply Chain Optimization

In [ ]:
# Import required libraries
import sys
sys.path.append('..')

from kg_model.connection import Neo4jConnection
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Database Connection Setup

In [ ]:
# Connect to Neo4j database
# Update these credentials as needed
neo4j_conn = Neo4jConnection(
    uri='bolt://localhost:7687',
    user='neo4j',
    password='password'
)
neo4j_conn.connect()
print('Connected to Neo4j database')

## 2. Single Points of Failure Analysis

Identify critical suppliers and products that could cause supply chain disruptions if they fail.

In [ ]:
# Query: Find products with only one supplier (Single Point of Failure)
spof_query = """
MATCH (p:Product)<-[:SUPPLIES]-(s:Supplier)
WITH p, count(s) as supplier_count
WHERE supplier_count = 1
MATCH (p)<-[r:SUPPLIES]-(s:Supplier)
RETURN p.product_id as product_id, 
       p.product_name as product_name,
       s.supplier_id as supplier_id,
       s.supplier_name as supplier_name,
       s.reliability_score as reliability
ORDER BY s.reliability_score
"""

spof_results = neo4j_conn.execute_query(spof_query)
spof_df = pd.DataFrame(spof_results)

print(f"\nFound {len(spof_df)} products with single supplier (SPOF):")
display(spof_df)

In [ ]:
# Query: Find critical suppliers (serving many products)
critical_suppliers_query = """
MATCH (s:Supplier)-[:SUPPLIES]->(p:Product)
WITH s, count(p) as product_count
WHERE product_count > 2
RETURN s.supplier_id as supplier_id,
       s.supplier_name as supplier_name,
       s.location as location,
       s.reliability_score as reliability,
       product_count
ORDER BY product_count DESC
"""

critical_suppliers = neo4j_conn.execute_query(critical_suppliers_query)
critical_df = pd.DataFrame(critical_suppliers)

print(f"\nFound {len(critical_df)} critical suppliers (serving >2 products):")
display(critical_df)

# Visualize
if len(critical_df) > 0:
    plt.figure(figsize=(12, 6))
    plt.bar(critical_df['supplier_name'], critical_df['product_count'])
    plt.xlabel('Supplier')
    plt.ylabel('Number of Products Supplied')
    plt.title('Critical Suppliers - Product Coverage')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 3. What-If Scenario Analysis

Analyze the impact of losing a supplier or route.

In [ ]:
# What-if: Simulate loss of a specific supplier
def analyze_supplier_loss(supplier_id):
    """Analyze impact of losing a supplier."""
    
    # Get affected products
    query = """
    MATCH (s:Supplier {supplier_id: $supplier_id})-[:SUPPLIES]->(p:Product)
    WITH p, s
    MATCH (p)<-[:SUPPLIES]-(alt:Supplier)
    WHERE alt.supplier_id <> s.supplier_id
    WITH p, s, count(alt) as alternative_count
    RETURN p.product_id as product_id,
           p.product_name as product_name,
           alternative_count
    ORDER BY alternative_count
    """
    
    results = neo4j_conn.execute_query(query, {'supplier_id': supplier_id})
    df = pd.DataFrame(results)
    
    print(f"\n=== Impact of losing supplier {supplier_id} ===")
    print(f"Products affected: {len(df)}")
    
    if len(df) > 0:
        at_risk = df[df['alternative_count'] == 0]
        print(f"Products with NO alternatives: {len(at_risk)}")
        
        if len(at_risk) > 0:
            print("\nProducts at risk (no alternatives):")
            display(at_risk)
        else:
            print("\nAll products have alternative suppliers!")
            display(df)
    
    return df

# Example: Analyze loss of supplier S003
impact_df = analyze_supplier_loss('S003')

In [ ]:
# What-if: Simulate route disruption
def analyze_route_disruption(route_id):
    """Analyze impact of route disruption."""
    
    query = """
    MATCH (r:Route {route_id: $route_id})<-[:USES_ROUTE]-(s:Supplier)
    MATCH (s)-[:SUPPLIES]->(p:Product)
    RETURN r.route_id as route_id,
           r.origin as origin,
           r.destination as destination,
           s.supplier_id as supplier_id,
           s.supplier_name as supplier_name,
           count(DISTINCT p) as products_affected
    """
    
    results = neo4j_conn.execute_query(query, {'route_id': route_id})
    df = pd.DataFrame(results)
    
    print(f"\n=== Impact of disrupting route {route_id} ===")
    
    if len(df) > 0:
        print(f"Route: {df.iloc[0]['origin']} -> {df.iloc[0]['destination']}")
        print(f"Suppliers affected: {len(df)}")
        print(f"Total products affected: {df['products_affected'].sum()}")
        display(df)
    else:
        print("No suppliers using this route")
    
    return df

# Example: Analyze disruption of route R001
route_impact_df = analyze_route_disruption('R001')

## 4. Risk Assessment Analysis

Identify and prioritize risks across the supply chain.

In [ ]:
# Query: High-risk suppliers and products
risk_query = """
MATCH (entity)-[hr:HAS_RISK]->(r:Risk)
WHERE r.severity >= 7 AND hr.impact_level = 'High'
RETURN labels(entity)[0] as entity_type,
       CASE 
         WHEN 'Product' IN labels(entity) THEN entity.product_id
         WHEN 'Supplier' IN labels(entity) THEN entity.supplier_id
         WHEN 'Route' IN labels(entity) THEN entity.route_id
       END as entity_id,
       r.risk_type as risk_type,
       r.severity as severity,
       r.probability as probability,
       hr.impact_level as impact_level
ORDER BY r.severity DESC, r.probability DESC
"""

risk_results = neo4j_conn.execute_query(risk_query)
risk_df = pd.DataFrame(risk_results)

print(f"\nFound {len(risk_df)} high-severity, high-impact risks:")
display(risk_df)

# Calculate risk score (severity * probability)
if len(risk_df) > 0:
    risk_df['risk_score'] = risk_df['severity'] * risk_df['probability']
    
    # Visualize risk distribution
    plt.figure(figsize=(10, 6))
    plt.scatter(risk_df['probability'], risk_df['severity'], 
                s=200, alpha=0.6, c=risk_df['risk_score'], cmap='YlOrRd')
    plt.xlabel('Probability')
    plt.ylabel('Severity')
    plt.title('Risk Assessment Matrix')
    plt.colorbar(label='Risk Score')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Risk summary by entity type
risk_summary_query = """
MATCH (entity)-[hr:HAS_RISK]->(r:Risk)
WITH labels(entity)[0] as entity_type, count(r) as risk_count
RETURN entity_type, risk_count
ORDER BY risk_count DESC
"""

risk_summary = neo4j_conn.execute_query(risk_summary_query)
risk_summary_df = pd.DataFrame(risk_summary)

print("\nRisk Distribution by Entity Type:")
display(risk_summary_df)

if len(risk_summary_df) > 0:
    plt.figure(figsize=(8, 6))
    plt.pie(risk_summary_df['risk_count'], labels=risk_summary_df['entity_type'], 
            autopct='%1.1f%%', startangle=90)
    plt.title('Risk Distribution by Entity Type')
    plt.axis('equal')
    plt.tight_layout()
    plt.show()

## 5. Supply Chain Optimization

Find optimization opportunities and alternative suppliers.

In [ ]:
# Query: Find products with cost optimization opportunities
cost_optimization_query = """
MATCH (p:Product)<-[r:SUPPLIES]-(s:Supplier)
WITH p, min(r.cost) as min_cost, max(r.cost) as max_cost, count(s) as supplier_count
WHERE supplier_count > 1 AND (max_cost - min_cost) > 5
RETURN p.product_id as product_id,
       p.product_name as product_name,
       supplier_count,
       min_cost,
       max_cost,
       (max_cost - min_cost) as cost_difference
ORDER BY cost_difference DESC
"""

cost_opt_results = neo4j_conn.execute_query(cost_optimization_query)
cost_opt_df = pd.DataFrame(cost_opt_results)

print(f"\nFound {len(cost_opt_df)} products with cost optimization opportunities:")
display(cost_opt_df)

if len(cost_opt_df) > 0:
    plt.figure(figsize=(12, 6))
    plt.bar(cost_opt_df['product_name'], cost_opt_df['cost_difference'])
    plt.xlabel('Product')
    plt.ylabel('Cost Difference ($)')
    plt.title('Cost Optimization Opportunities')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
# Query: Supplier diversity analysis
diversity_query = """
MATCH (s:Supplier)
WITH s.location as location, count(s) as supplier_count
RETURN location, supplier_count
ORDER BY supplier_count DESC
"""

diversity_results = neo4j_conn.execute_query(diversity_query)
diversity_df = pd.DataFrame(diversity_results)

print("\nSupplier Geographic Diversity:")
display(diversity_df)

if len(diversity_df) > 0:
    plt.figure(figsize=(10, 6))
    plt.barh(diversity_df['location'], diversity_df['supplier_count'])
    plt.xlabel('Number of Suppliers')
    plt.ylabel('Location')
    plt.title('Supplier Geographic Distribution')
    plt.tight_layout()
    plt.show()

## 6. Summary and Recommendations

In [ ]:
# Generate summary report
print("="*60)
print("SUPPLY CHAIN RESILIENCE ANALYSIS SUMMARY")
print("="*60)

# Count nodes
node_counts = neo4j_conn.execute_query("""
MATCH (p:Product) WITH count(p) as products
MATCH (s:Supplier) WITH products, count(s) as suppliers
MATCH (r:Route) WITH products, suppliers, count(r) as routes
MATCH (risk:Risk) WITH products, suppliers, routes, count(risk) as risks
RETURN products, suppliers, routes, risks
""")

if node_counts:
    counts = node_counts[0]
    print(f"\nKnowledge Graph Statistics:")
    print(f"  - Products: {counts['products']}")
    print(f"  - Suppliers: {counts['suppliers']}")
    print(f"  - Routes: {counts['routes']}")
    print(f"  - Risks: {counts['risks']}")

print(f"\nResilience Findings:")
print(f"  - Single Points of Failure: {len(spof_df)}")
print(f"  - Critical Suppliers: {len(critical_df)}")
print(f"  - High-Severity Risks: {len(risk_df)}")
print(f"  - Cost Optimization Opportunities: {len(cost_opt_df)}")

print("\nRecommendations:")
if len(spof_df) > 0:
    print(f"  1. Address {len(spof_df)} single points of failure by diversifying suppliers")
if len(risk_df) > 0:
    print(f"  2. Develop mitigation plans for {len(risk_df)} high-severity risks")
if len(critical_df) > 0:
    print(f"  3. Monitor {len(critical_df)} critical suppliers closely")
if len(cost_opt_df) > 0:
    print(f"  4. Explore cost optimization for {len(cost_opt_df)} products")

print("\n" + "="*60)

In [ ]:
# Clean up
neo4j_conn.close()
print('\nConnection closed')